# MATRIX Pilot #1 — GridWorld Agent Learning

Этот ноутбук демонстрирует обучение MPDT-агента в среде GridWorld с помощью генетического алгоритма.

**Стек:** Java 25 + Quarkus 3.35.4, MPDT-нейроны, генетический алгоритм.

**Запуск:** `java -jar matrix-core.jar simulate -g 100 -p 30 -k 16 --seed 42`

## 1. Генерация данных

Запустим симуляцию и сохраним метрики в JSON.

In [ ]:
import subprocess, json, os

generations = 50
population = 20
k = 16
seed = 42

result = subprocess.run([
    "java", "-jar", "matrix-core/build/matrix-core-*-runner.jar",
    "simulate",
    "-g", str(generations),
    "-p", str(population),
    "-k", str(k),
    "--seed", str(seed)
], capture_output=True, text=True, shell=True)

lines = result.stderr.split('\n')
gen_data = []
for line in lines:
    if "Gen" in line and "best=" in line:
        parts = line.strip().split()
        gen_data.append({
            "generation": int(parts[1]),
            "best": int(parts[3].split("=")[1]),
            "avg": int(parts[4].split("=")[1])
        })
print(f"Loaded {len(gen_data)} generations of data")
gen_data[:3]

## 2. График фитнеса

Как эволюционирует фитнес агента с поколениями.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

gens = [d["generation"] for d in gen_data]
best = [d["best"] for d in gen_data]
avg = [d["avg"] for d in gen_data]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(gens, best, 'b-', linewidth=2, label='Best Fitness')
ax.plot(gens, avg, 'orange', linewidth=1.5, alpha=0.7, label='Average Fitness')
ax.fill_between(gens, avg, best, alpha=0.1, color='blue')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Generation')
ax.set_ylabel('Fitness Score')
ax.set_title('MPDT Agent Evolution in GridWorld\n(20 agents × 16 inputs)', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

improvement = best[-1] - best[0]
ax.annotate(f'+{improvement} points', xy=(gens[-1], best[-1]),
            xytext=(gens[-1]-10, best[-1]+10),
            arrowprops=dict(arrowstyle='->', color='green'),
            fontsize=11, color='green')

plt.tight_layout()
plt.savefig('gridworld_evolution.png', dpi=150)
plt.show()
print(f"Fitness improvement: {best[0]} → {best[-1]} (+{improvement})")

## 3. Анализ стабильности

Проверим, насколько стабильно обучение при разных seed.

In [ ]:
seeds = [42, 123, 456, 789, 1024]
all_runs = []

for s in seeds:
    r = subprocess.run([
        "java", "-jar", "matrix-core/build/matrix-core-*-runner.jar",
        "simulate", "-g", "30", "-p", "20", "-k", "16",
        "--seed", str(s)
    ], capture_output=True, text=True, shell=True)
    final_best = None
    for line in r.stderr.split('\n'):
        if "Gen" in line and "best=" in line:
            parts = line.strip().split()
            g = int(parts[1])
            b = int(parts[3].split("=")[1])
            if g == 30:
                final_best = b
    all_runs.append(final_best)

print(f"Final fitness across {len(seeds)} seeds:")
for s, f in zip(seeds, all_runs):
    print(f"  seed={s}: {f}")
print(f"\nMean: {np.mean(all_runs):.0f}  Std: {np.std(all_runs):.0f}")

## 4. Чего мы достигли

- **Не градиентный спуск:** обучение идёт через эволюционный алгоритм.
- **Интерпретируемость:** поведение агента — дерево решений, можно прочитать.
- **Энергоэффективность:** инференс нейрона — наносекунды.
- **Детерминизм:** одинаковый seed = одинаковый результат (воспроизводимость).

*Следующий пилот: проактивный чат-бот с Этическим фильтром.*